In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import joblib

print("Import Sucess")


Import Sucess


In [2]:
RNG = np.random.default_rng(42)

def temp_factor(t):
    return float (np.exp(-((t - 29.5) ** 2) / (2 * (4.0 ** 2))))

def ph_factor(ph):
    if 7.0 <= ph <= 8.5:
        return 1.0
    if ph < 7.0:
        return max(0.0, 1.0 -(7.0 - ph) * 0.35)
    return max(0.0, 1.0 - (ph - 8.5) * 0.30)

def tds_factor(tds):
    if 100 <= tds <= 400:
        return 1.0
    if tds < 100:
        return max(0.0, 0.7 + (tds / 100) * 0.3)
    return max(0.0, 1.0 - (tds - 400) * 0.002)

def do_factor(do):
    if do >= 5.0:
        return 1.0
    return max(0.0, do / 5.0)

def turbidity_factor(ntu):
    if ntu <= 25:
        return 1.0
    return max(0.0, 1.0 - (ntu - 25) * 0.02)

def waterlevel_factor (wl):
    if 0.5 <= wl <= 2.0:
        return 1.0
    if wl < 0.5:
        return max (0.0, wl /0.5)
    return max(0.0, 1.0 - (wl - 2.0) * 0.5)

def maturity_factor (weight):
    if weight < 25:
        return 1.0
    return max (0.4, 1.0 - (weight - 25) * 0.03)

for temp in [24, 27, 29.5, 32, 34]:
    print(f" Temp {temp}C -> multiplier {temp_factor(temp):.3f}")

for tds in [50, 150, 300, 500, 700]:
    print(f" TDS {tds}ppm -> multiplier {tds_factor(tds):.3f}")

 Temp 24C -> multiplier 0.389
 Temp 27C -> multiplier 0.823
 Temp 29.5C -> multiplier 1.000
 Temp 32C -> multiplier 0.823
 Temp 34C -> multiplier 0.531
 TDS 50ppm -> multiplier 0.850
 TDS 150ppm -> multiplier 1.000
 TDS 300ppm -> multiplier 1.000
 TDS 500ppm -> multiplier 0.800
 TDS 700ppm -> multiplier 0.400


In [3]:
# Section 2.5 — Feeding Response Curve (galing sa literature)
# Basehan:
#   Gupta et al. — M. rosenbergii fed 10% -> 7% -> 5% ng body weight/araw,
#     bumababa habang lumalaki (juvenile mas mataas, malaki mas mababa)
#     https://www.academia.edu/7062101/
#   Starvation study — kaunting bawas (1-2 araw) = mas MAGANDANG FCR at survival
#     kaysa tuloy-tuloy na pakain; sobra = sayang (non-linear)
#     https://www.researchgate.net/publication/312447397

def optimal_feed_rate(weight):
    """Optimal feed rate (% body weight/araw) na naka-scale sa laki.
    Gupta et al.: 10% (maliit) -> 5% (malaki). Linear interpolation."""
    if weight <= 2:
        return 10.0      # juvenile — mataas
    if weight >= 25:
        return 5.0       # malapit harvest — mababa
    # linear mula 10% (2g) hanggang 5% (25g)
    return 10.0 - (weight - 2) * (5.0 / 23.0)

def feed_factor(feed_rate, weight):
    """0-1 multiplier: gaano ka-optimal ang feeding rate para sa laki na 'to.
    Peak sa optimal rate, penalty kapag kulang o sobra."""
    opt = optimal_feed_rate(weight)
    if feed_rate < opt * 0.6:          # matinding underfeed (<60% ng optimal)
        return max(0.4, feed_rate / (opt * 0.6) * 0.7)
    if feed_rate < opt:                 # bahagyang underfeed
        return 0.7 + (feed_rate - opt * 0.6) / (opt * 0.4) * 0.3
    if feed_rate <= opt * 1.4:          # optimal band (hanggang +40%)
        return 1.0
    # overfeed — bahagyang penalty (starvation study: sobra = sayang)
    return max(0.6, 1.0 - (feed_rate - opt * 1.4) * 0.03)

print("Feed response curve defined!")

# Test — dapat peak sa optimal, penalty sa dulo
print("\nMaliit na ulang (5g, optimal ~9.3%):")
for fr in [3, 6, 9, 12, 16]:
    print(f"  Feed {fr}% -> factor {feed_factor(fr, 5):.3f}")

print("\nMalaki na ulang (25g, optimal 5%):")
for fr in [2, 4, 5, 7, 10]:
    print(f"  Feed {fr}% -> factor {feed_factor(fr, 25):.3f}")

Feed response curve defined!

Maliit na ulang (5g, optimal ~9.3%):
  Feed 3% -> factor 0.400
  Feed 6% -> factor 0.731
  Feed 9% -> factor 0.972
  Feed 12% -> factor 1.000
  Feed 16% -> factor 0.913

Malaki na ulang (25g, optimal 5%):
  Feed 2% -> factor 0.467
  Feed 4% -> factor 0.850
  Feed 5% -> factor 1.000
  Feed 7% -> factor 1.000
  Feed 10% -> factor 0.910


In [4]:
# Section 3 — Cycle Simulation (may feeding na)

def sample_week_conditions():
    return {
        "avgWaterTemp":       float(np.clip(RNG.normal(29.0, 2.2), 22, 35)),
        "avgPh":              float(np.clip(RNG.normal(7.6, 0.55), 6.0, 9.5)),
        "avgDissolvedOxygen": float(np.clip(RNG.normal(6.2, 1.1), 2.0, 9.0)),
        "avgTds":             float(np.clip(RNG.normal(250, 90), 30, 700)),
        "avgTurbidity":       float(np.clip(RNG.normal(15.0, 8.0), 0.0, 60.0)),
        "avgWaterLevel":      float(np.clip(RNG.normal(1.3, 0.3), 0.3, 2.2)),
    }

def sample_feed_rate(weight):
    """Aktwal na feed rate — karaniwan malapit sa optimal, may variation.
    Realistic: minsan kulang/sobra ang farmer sa pagpapakain."""
    opt = optimal_feed_rate(weight)
    # normal distribution centered sa optimal, ±25% variation
    return float(np.clip(RNG.normal(opt, opt * 0.25), 1.0, 15.0))

BASE_GROWTH = 2.2      # g/week sa optimal conditions
WEEKS = 18
N_CYCLES = 45

def simulate():
    rows = []
    for cycle in range(1, N_CYCLES + 1):
        weight = float(np.clip(RNG.normal(1.5, 0.4), 0.5, 3.0))
        for week in range(1, WEEKS + 1):
            c = sample_week_conditions()
            feed_rate = sample_feed_rate(weight)   # BAGO
            mult = (
                temp_factor(c["avgWaterTemp"]) *
                ph_factor(c["avgPh"]) *
                tds_factor(c["avgTds"]) *
                do_factor(c["avgDissolvedOxygen"]) *
                turbidity_factor(c["avgTurbidity"]) *
                waterlevel_factor(c["avgWaterLevel"]) *
                feed_factor(feed_rate, weight) *    # BAGO
                maturity_factor(weight)
            )
            noise = 1.0 + RNG.normal(0, 0.08)
            growth = max(0.0, BASE_GROWTH * mult * noise)

            rows.append({
                "cycleId": cycle,
                "weekNumber": week,
                "currentWeight": round(weight, 3),
                **{k: round(v, 3) for k, v in c.items()},
                "avgFeedRate": round(feed_rate, 3),   # BAGO — feature
                "weeklyGrowthIncrement": round(growth, 3),
            })
            weight += growth
    return pd.DataFrame(rows)

df = simulate()
print(f"Nagawa: {len(df)} rows across {N_CYCLES} cycles\n")
print(df.head(10).to_string(index=False))
print(f"\nFeed rate range: {df['avgFeedRate'].min():.1f}% - {df['avgFeedRate'].max():.1f}%")
print(f"Weekly growth mean: {df['weeklyGrowthIncrement'].mean():.3f} g")

Nagawa: 810 rows across 45 cycles

 cycleId  weekNumber  currentWeight  avgWaterTemp  avgPh  avgDissolvedOxygen  avgTds  avgTurbidity  avgWaterLevel  avgFeedRate  weeklyGrowthIncrement
       1           1          1.622        26.712  8.013               7.235  74.407         4.583          1.338        9.209                  1.497
       1           2          3.118        27.123  8.084               7.056 255.943        24.018          1.440        7.661                  1.593
       1           3          4.711        26.890  8.083               6.145 233.362         9.553          1.667        9.047                  1.668
       1           4          6.379        28.225  7.893               6.602 287.146        18.447          1.942        8.129                  1.853
       1           5          8.231        27.210  7.939               7.442 239.745         8.279          1.053       10.052                  1.978
       1           6         10.210        30.195  7.234         

In [5]:
# Section 4 — Train Random Forest (WALANG water level, 8 features)
FEATURES = [
    "weekNumber", "currentWeight",
    "avgWaterTemp", "avgPh", "avgDissolvedOxygen",
    "avgTds", "avgTurbidity",
    "avgFeedRate",   # tinanggal ang avgWaterLevel
]
TARGET = "weeklyGrowthIncrement"

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f"Dataset: {len(df)} rows | train={len(X_train)} test={len(X_test)}")
print(f"Features: {len(FEATURES)} (tinanggal ang water level)\n")

rf = RandomForestRegressor(n_estimators=150, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

mlr = LinearRegression()
mlr.fit(X_train, y_train)

def evaluate(name, model):
    pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)
    print(f"  {name:<24} RMSE={rmse:.4f}  MAE={mae:.4f}  R2={r2:.4f}")
    return r2

print("Test-set performance")
print("-" * 55)
rf_r2 = evaluate("Random Forest", rf)
mlr_r2 = evaluate("MLR (baseline)", mlr)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv = cross_val_score(rf, X_train, y_train, cv=kf, scoring="r2")
print(f"\n5-fold CV R2: mean={cv.mean():.4f} (+/- {cv.std():.4f})")

print("\nFeature importance:")
print("-" * 55)
imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
for feat, val in imp.items():
    bar = "#" * int(val * 50)
    print(f"  {feat:<20} {val:.3f}  {bar}")

print(f"\nRF R2 >= 0.80? {'OO' if rf_r2 >= 0.80 else 'HINDI'}")
print(f"RF > MLR?      {'OO' if rf_r2 > mlr_r2 else 'HINDI'}")

Dataset: 810 rows | train=648 test=162
Features: 8 (tinanggal ang water level)

Test-set performance
-------------------------------------------------------
  Random Forest            RMSE=0.1990  MAE=0.1587  R2=0.8222
  MLR (baseline)           RMSE=0.4317  MAE=0.3384  R2=0.1630

5-fold CV R2: mean=0.7406 (+/- 0.0178)

Feature importance:
-------------------------------------------------------
  avgWaterTemp         0.472  #######################
  avgFeedRate          0.288  ##############
  currentWeight        0.062  ###
  avgDissolvedOxygen   0.054  ##
  avgPh                0.039  #
  avgTurbidity         0.036  #
  avgTds               0.034  #
  weekNumber           0.015  

RF R2 >= 0.80? OO
RF > MLR?      OO


In [ ]:
# Section 5 — Save Model + Yield Projection (8 features, walang water level)
import joblib

joblib.dump(rf, "rf_growth_model.joblib")
print("Model saved -> rf_growth_model.joblib\n")

def project_harvest_weight(start_week, current_weight, conditions, feed_rate_pct,
                            harvest_week=18):
    weight = current_weight
    for wk in range(start_week, harvest_week + 1):
        feat = pd.DataFrame([{
            "weekNumber": wk,
            "currentWeight": weight,
            "avgWaterTemp": conditions["avgWaterTemp"],
            "avgPh": conditions["avgPh"],
            "avgDissolvedOxygen": conditions["avgDissolvedOxygen"],
            "avgTds": conditions["avgTds"],
            "avgTurbidity": conditions["avgTurbidity"],
            "avgFeedRate": feed_rate_pct,
            # tinanggal ang avgWaterLevel
        }])
        growth = rf.predict(feat)[0]
        weight += growth
    return weight

optimal_conditions = {
    "avgWaterTemp": 29.0,
    "avgPh": 7.6,
    "avgDissolvedOxygen": 6.5,
    "avgTds": 250,
    "avgTurbidity": 12,
    # tinanggal ang avgWaterLevel
}

projected_weight = project_harvest_weight(
    start_week=8, current_weight=14.0,
    conditions=optimal_conditions, feed_rate_pct=6.0,
    harvest_week=18
)

print("YIELD PROJECTION")
print("-" * 45)
print(f"  Kasalukuyan: Week 8, 14.0g")
print(f"  Projected harvest weight (Week 18): {projected_weight:.1f}g")

initial_stock = 50
survival_rate = 80
projected_yield = initial_stock * (survival_rate/100) * projected_weight / 1000
print(f"\n  YIELD = {initial_stock} x {survival_rate}% x {projected_weight:.1f}g / 1000")
print(f"  YIELD = {projected_yield:.2f} kg")

Model saved -> rf_growth_model.joblib



ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- avgWaterLevel
